In [2]:
import pandas as pd
import json
import re
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # loads the .env file

True

In [3]:
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2025-01-01-preview"
)

In [4]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Say hello in one word"}],
    max_tokens=10,
)
print(response.choices[0].message.content)

Hello!


In [5]:
df = pd.read_csv("data/Reddit_askdocs_2k.csv")
df["question_text"] = df["title"].fillna("").astype(str) + " " + df["selftext"].fillna("").astype(str)
sample_df = df.sample(n=25, random_state=42).reset_index(drop=True)
print(f"Loaded {len(sample_df)} questions")
sample_df["question_text"].head(3)

Loaded 25 questions


0    I used to be malnourished for 1.5 years as an ...
1    My dad (70M) was informed that he needs to sta...
2    A question about anesthesia  When getting a pl...
Name: question_text, dtype: object

In [6]:
import sys
print(sys.version)  # should show 3.9.x

import scispacy
import spacy
nlp = spacy.load("en_core_sci_sm")
print("scispaCy works in notebook!")

3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 18:02:02) 
[Clang 18.1.8 ]
scispaCy works in notebook!


In [7]:
def extract_medical_entities(text):
    doc = nlp(text)
    entities = [ent.text.strip() for ent in doc.ents if len(ent.text.strip()) > 2]
    return entities

In [8]:
test_text = sample_df["question_text"].iloc[0]
print("Question:", test_text[:300])
print("\nExtracted entities:", extract_medical_entities(test_text))

Question: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I'm a male who is 16 and 5'7", and my parents are 5'9.5" and 5'2.5".

Extracted entities: ['years', 'early teen', "I'm", 'eating', 'will I permanently stay', 'height', 'male', 'parents', "5'2.5"]


In [18]:
def find_medical_relationships(question_text, entities):
    if len(entities) < 2:
        return []
    
    prompt = f"""You are a strict medical NLP assistant. Your job is to find ONLY real, specific medical relationship pairs from a list of terms.

ALLOWED relationship types (use exactly these labels):
- "drug-disease": a specific drug/medication treats a specific disease
- "disease-symptom": a specific disease causes a specific symptom
- "symptom-disease": a specific symptom indicates a specific disease

STRICT RULES:
1. Both terms MUST be real, specific medical entities:
   - VALID: disease names, symptom names, drug names
   - INVALID: test results (positive, negative, normal), procedures (ultrasound, MRI), vague words (issues, problems, positive, normal, stuff), body parts alone (kidney, chest), emotions alone (scared, worried)
2. The two terms must NOT be the same thing worded differently (e.g. "allergy" and "allergies", "rash" and "heat rash")
3. Do NOT create reverse pairs — if you write (A, B), do NOT also write (B, A)
4. If fewer than 2 valid medical entities exist, return empty list
5. Only pair things with a DIRECT medical relationship — not just because they appear together

GOOD examples:
- ["diabetes", "kidney failure", "disease-symptom"] ✓
- ["metformin", "diabetes", "drug-disease"] ✓
- ["chest pain", "heart attack", "symptom-disease"] ✓

BAD examples (do not do these):
- ["covid", "positive", "disease-symptom"] ✗ (positive is not a symptom)
- ["allergy", "allergies", "disease-symptom"] ✗ (same thing)
- ["rash", "rash", "disease-symptom"] ✗ (same thing)
- ["flu", "runny nose", "disease-symptom"] AND ["runny nose", "flu", "symptom-disease"] ✗ (reverse duplicate)

Terms extracted: {entities}

Original question for context: \"\"\"{question_text[:400]}\"\"\"

Return ONLY valid JSON, no markdown:
{{"pairs": [["term1", "term2", "relationship_type"], ...]}}
"""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=400,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    result = json.loads(raw)
    return result.get("pairs", [])

In [19]:
import time

results = []

for i, row in sample_df.iterrows():
    text = row["question_text"]
    entities = extract_medical_entities(text)
    try:
        pairs = find_medical_relationships(text, entities)
    except Exception as e:
        pairs = []
        print(f"Row {i} error: {e}")
    
    results.append({
        "question_text": text,
        "entities": entities,
        "relationship_pairs": pairs
    })
    
    time.sleep(0.5)
    print(f"✓ {i+1}/25")

results_df = pd.DataFrame(results)
print("\nDone!")

✓ 1/25
✓ 2/25
✓ 3/25
✓ 4/25
✓ 5/25
✓ 6/25
✓ 7/25
✓ 8/25
✓ 9/25
✓ 10/25
✓ 11/25
✓ 12/25
✓ 13/25
✓ 14/25
✓ 15/25
✓ 16/25
✓ 17/25
✓ 18/25
✓ 19/25
✓ 20/25
✓ 21/25
✓ 22/25
✓ 23/25
✓ 24/25
✓ 25/25

Done!


In [23]:
for i, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Question {i+1}: {row['question_text'][:150]}")
    print(f"Pairs: {row['relationship_pairs']}")


Question 1: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I
Pairs: []

Question 2: My dad (70M) was informed that he needs to start dialysis. Anything we can do? My dad who is (6feet tall and about 260lbs) came home crying that his d
Pairs: [['diabetes', 'dialysis', 'disease-symptom']]

Question 3: A question about anesthesia  When getting a planned procedure, you're told to fast the night before as to not risk aspiration. Done that twice, once w
Pairs: []

Question 4: Hi! I (26M) cut my finger and lost feeling in the tip, what are the chances of the nerve repairing itself? 
My finger got hit by a crazy sharp knife t
Pairs: []

Question 5: Is it crazy to stay in a hotel to avoid family members with covid? I m27 am vaccinated. Both of my parents who I live with (also vaccinated) have covi
Pairs: [['covid', 'isolating', 'disease-symptom']]

Question 6: pulsating in my right collarbone,